# 19 — Caching and Cost Optimization

In-memory caching, token budgeting, and prompt compression.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

## Example 1: In-Memory Cache

In [ ]:
set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = ChatPromptTemplate.from_template("What is the capital of {country}?") | llm | StrOutputParser()

for country in ["France", "Japan", "France", "Japan", "Brazil"]:
    start = time.time()
    result = chain.invoke({"country": country})
    elapsed = time.time() - start
    cached = "CACHED" if elapsed < 0.1 else "API"
    print(f"[{cached}] {country}: {result} ({elapsed:.3f}s)")

set_llm_cache(None)

## Example 2: Token Budgeting

In [ ]:
def estimate_tokens(text): return len(text) // 4
def truncate_to_budget(text, max_tokens):
    if estimate_tokens(text) <= max_tokens: return text
    return text[:max_tokens * 4] + "... [truncated]"

long_doc = ("LangChain is a framework for developing applications powered by language models. " * 20)
for budget in [50, 100, 200]:
    truncated = truncate_to_budget(long_doc, budget)
    result = (ChatPromptTemplate.from_template("Summarise in one sentence:\n\n{text}") | llm | StrOutputParser()).invoke({"text": truncated})
    print(f"Budget: {budget} tokens | ~{estimate_tokens(truncated)} used\nSummary: {result}\n")

## Example 3: Prompt Compression

In [ ]:
compress_chain = ChatPromptTemplate.from_template(
    "Compress this text to be as short as possible while keeping all key information.\n\nOriginal:\n{text}\n\nCompressed:"
) | llm | StrOutputParser()

for original in [
    "I would really appreciate it if you could please help me understand what exactly the concept of machine learning is all about.",
    "Could you please provide me with a comprehensive explanation of the differences between Python lists and tuples?",
]:
    compressed = compress_chain.invoke({"text": original})
    savings = (1 - estimate_tokens(compressed) / estimate_tokens(original)) * 100
    print(f"Original ({estimate_tokens(original)} tok): {original[:60]}...")
    print(f"Compressed ({estimate_tokens(compressed)} tok): {compressed}")
    print(f"Savings: {savings:.0f}%\n")